In [1]:
import pandas as pd

# Create a fictional customer dataset for testing
data = {
    "customer_id": [101, 102, 103, 104, 104, 106, 107, 108],
    "name": [
        "Alice Mwangi", "Brian Otieno", "Carol Njeri", "David Kamau",
        "David Kamau", "Faith Wanjiku", "", "Grace Achieng"
    ],
    "email": [
        "alice@example.com", "brian@example.com", "invalid-email",
        "david@example.com", "david@example.com", "faith@example.com",
        "john@example.com", "grace@example.com"
    ],
    "age": [24, 31, 28, 45, 45, None, 22, 150],
    "city": [
        "Nairobi", "Mombasa", "Kisumu", "Nairobi",
        "Nairobi", "Nakuru", "Eldoret", "Kisumu"
    ]
}

df = pd.DataFrame(data)

print("DATA QUALITY AUDIT PROJECT")
print("=" * 35)
print(f"Dataset rows: {len(df)}")
print(f"Dataset columns: {len(df.columns)}")
display(df)

DATA QUALITY AUDIT PROJECT
Dataset rows: 8
Dataset columns: 5


,customer_id,name,email,age,city
0,101,Alice Mwangi,alice@example.com,24.0,Nairobi
1,102,Brian Otieno,brian@example.com,31.0,Mombasa
2,103,Carol Njeri,invalid-email,28.0,Kisumu
3,104,David Kamau,david@example.com,45.0,Nairobi
4,104,David Kamau,david@example.com,45.0,Nairobi
5,106,Faith Wanjiku,faith@example.com,NaN,Nakuru
6,107,,john@example.com,22.0,Eldoret
7,108,Grace Achieng,grace@example.com,150.0,Kisumu


In [2]:

# DATA QUALITY AUDIT

# 1. Check for missing values
missing_values = df.isnull().sum()

# 2. Check for blank text fields
blank_names = df["name"].fillna("").str.strip().eq("").sum()

# 3. Check for duplicate customer IDs
duplicate_ids = df["customer_id"].duplicated(keep=False).sum()

# 4. Check for duplicate complete records
duplicate_records = df.duplicated().sum()

# 5. Check email format
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
invalid_emails = ~df["email"].fillna("").str.match(email_pattern)

# 6. Check for unrealistic ages (assumed valid range: 0–120)
invalid_ages = df["age"].notna() & ~df["age"].between(0, 120)

# Print the audit report
print("DATA QUALITY AUDIT REPORT")
print("=" * 35)
print("\nMissing values by column:")
print(missing_values)

print(f"\nBlank names: {blank_names}")
print(f"Rows involved in duplicate customer IDs: {duplicate_ids}")
print(f"Duplicate complete records: {duplicate_records}")
print(f"Invalid email rows: {invalid_emails.sum()}")
print(f"Invalid age rows: {invalid_ages.sum()}")


DATA QUALITY AUDIT REPORT

Missing values by column:
customer_id    0
name           0
email          0
age            1
city           0
dtype: int64

Blank names: 1
Rows involved in duplicate customer IDs: 2
Duplicate complete records: 1
Invalid email rows: 1
Invalid age rows: 1


In [3]:

# CLEAN THE DATASET

clean_df = df.copy()

# Record the original row count
original_rows = len(clean_df)

# 1. Remove exact duplicate records
clean_df = clean_df.drop_duplicates()

# 2. Standardize text fields by trimming extra spaces
for column in ["name", "email", "city"]:
    clean_df[column] = clean_df[column].astype("string").str.strip()

# 3. Convert blank names and emails into missing values
clean_df["name"] = clean_df["name"].replace("", pd.NA)
clean_df["email"] = clean_df["email"].replace("", pd.NA)

# 4. Flag invalid emails
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
clean_df["invalid_email"] = ~clean_df["email"].fillna("").str.match(email_pattern)

# 5. Flag ages outside the assumed valid range of 0–120
clean_df["invalid_age"] = (
    clean_df["age"].notna()
    & ~clean_df["age"].between(0, 120)
)

# 6. Flag duplicate customer IDs without deleting them automatically
clean_df["duplicate_customer_id"] = clean_df["customer_id"].duplicated(keep=False)

print("CLEANING SUMMARY")
print("=" * 30)
print(f"Original rows: {original_rows}")
print(f"Rows after exact-duplicate removal: {len(clean_df)}")
print(f"Rows removed: {original_rows - len(clean_df)}")
print("\nREVIEW FLAGS")
display(clean_df)


CLEANING SUMMARY
Original rows: 8
Rows after exact-duplicate removal: 7
Rows removed: 1

REVIEW FLAGS


,customer_id,name,email,age,city,invalid_email,invalid_age,duplicate_customer_id
0,101,Alice Mwangi,alice@example.com,24.0,Nairobi,False,False,False
1,102,Brian Otieno,brian@example.com,31.0,Mombasa,False,False,False
2,103,Carol Njeri,invalid-email,28.0,Kisumu,True,False,False
3,104,David Kamau,david@example.com,45.0,Nairobi,False,False,False
5,106,Faith Wanjiku,faith@example.com,NaN,Nakuru,False,False,False
6,107,<NA>,john@example.com,22.0,Eldoret,False,False,False
7,108,Grace Achieng,grace@example.com,150.0,Kisumu,False,True,False


In [4]:

from pathlib import Path

# Create an output folder
output_dir = Path("reports")
output_dir.mkdir(exist_ok=True)

# Recalculate duplicate IDs from the ORIGINAL data
duplicate_id_count = df["customer_id"].duplicated(keep=False).sum()

# Build a concise audit summary
audit_summary = pd.DataFrame({
    "metric": [
        "Original row count",
        "Rows after exact duplicate removal",
        "Exact duplicate rows removed",
        "Rows involved in duplicate customer IDs",
        "Missing age values",
        "Blank names",
        "Invalid email rows",
        "Invalid age rows"
    ],
    "count": [
        len(df),
        len(clean_df),
        len(df) - len(clean_df),
        duplicate_id_count,
        df["age"].isna().sum(),
        df["name"].fillna("").str.strip().eq("").sum(),
        invalid_emails.sum(),
        invalid_ages.sum()
    ]
})

# Export the cleaned data and audit summary
clean_df.to_csv(output_dir / "cleaned_customer_data.csv", index=False)
audit_summary.to_csv(output_dir / "audit_summary.csv", index=False)

print("FILES CREATED")
print("=" * 30)
print(f"Cleaned dataset: {output_dir / 'cleaned_customer_data.csv'}")
print(f"Audit summary: {output_dir / 'audit_summary.csv'}")
display(audit_summary)


FILES CREATED
Cleaned dataset: reports/cleaned_customer_data.csv
Audit summary: reports/audit_summary.csv


,metric,count
0,Original row count,8
1,Rows after exact duplicate removal,7
2,Exact duplicate rows removed,1
3,Rows involved in duplicate customer IDs,2
4,Missing age values,1
5,Blank names,1
6,Invalid email rows,1
7,Invalid age rows,1
